In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Machine Learning

✅ Part 1: Data Preprocessing
Tasks:

Remove 0-byte files

Remove duplicates

Load & zero-pad signals to 16,000 samples

In [ ]:
import os
import librosa
import numpy as np
import hashlib
from tqdm import tqdm


def compute_md5(file_path):
    with open(file_path, 'rb') as f:
        return hashlib.md5(f.read()).hexdigest()


def load_and_pad(file_path, sr=16000):
    try:
        y, _ = librosa.load(file_path, sr=sr)
        if len(y) < sr:
            y = np.pad(y, (0, sr - len(y)))
        return y
    except Exception:
        return None


def _infer_label_from_path(path_parts_lower: list[str]):
    # Try common conventions across audio deepfake datasets
    for p in path_parts_lower:
        if p in {"real", "bonafide", "bona-fide", "genuine", "original"}:
            return 0
        if p in {"fake", "spoof", "deepfake", "synth", "synthetic", "generated"}:
            return 1
    return None


def preprocess_audio_dir(base_path, sr=16000, audio_exts=(".wav", ".flac", ".mp3", ".ogg", ".m4a")):
    """Load audio under base_path and infer labels from folder names.

    Expected (flexible) layout examples:
    - base_path/real/*.wav and base_path/fake/*.wav
    - base_path/Real/** and base_path/Fake/**

    If labels can't be inferred, files are skipped.
    """
    all_audio = []
    all_labels = []
    seen_hashes = set()

    audio_files = []
    for root, _, files in os.walk(base_path):
        for fn in files:
            if os.path.splitext(fn)[1].lower() in audio_exts:
                audio_files.append(os.path.join(root, fn))

    if not audio_files:
        raise FileNotFoundError(
            f"No audio files found under: {base_path}. "
            "Check FAKEAUDIO_DIR (it should point to the dataset root folder)."
        )

    for path in tqdm(audio_files, desc=f"Scanning {os.path.basename(str(base_path).rstrip('/')) or base_path}"):
        try:
            if os.path.getsize(path) == 0:
                continue
        except OSError:
            continue

        parts = [p.lower() for p in os.path.normpath(path).split(os.sep)]
        label = _infer_label_from_path(parts)
        if label is None:
            continue

        file_hash = compute_md5(path)
        if file_hash in seen_hashes:
            continue
        seen_hashes.add(file_hash)

        y = load_and_pad(path, sr=sr)
        if y is None:
            continue

        all_audio.append(y)
        all_labels.append(label)

    return all_audio, all_labels


✅ Part 2: Feature Extraction (MFCC + Standardization)

In [ ]:
from sklearn.preprocessing import StandardScaler

def extract_mfcc_features(audio_list, sr=16000, n_mfcc=40):
    mfcc_features = []
    for y in tqdm(audio_list, desc="Extracting MFCCs"):
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
        mfcc_mean = np.mean(mfcc.T, axis=0)
        mfcc_features.append(mfcc_mean)
    return np.array(mfcc_features)

def normalize_features(X_train, X_val, X_test):
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val = scaler.transform(X_val)
    X_test = scaler.transform(X_test)
    return X_train, X_val, X_test, scaler


✅ Part 3: Classification Models (Train & Predict)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier

def train_models(X_train, y_train):
    models = {
        "Random Forest": RandomForestClassifier(),
        "SVM": SVC(probability=True),
        "MLP": MLPClassifier(hidden_layer_sizes=(100,), max_iter=500),
        "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss')
    }

    trained = {}
    for name, model in models.items():
        model.fit(X_train, y_train)
        trained[name] = model
    return trained


✅ Part 4: Evaluate on All 4 Datasets
You’ll loop over 4 dataset folders:

for-2seconds

for-norm

for-original

for-rerec

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

def evaluate_models(models, X_test, y_test):
    results = {}
    for name, model in models.items():
        y_pred = model.predict(X_test)
        acc = accuracy_score(y_test, y_pred)
        report = classification_report(y_test, y_pred, output_dict=True)
        results[name] = {
            "accuracy": acc,
            "precision": report['weighted avg']['precision'],
            "recall": report['weighted avg']['recall'],
            "f1": report['weighted avg']['f1-score']
        }
    return results


✅ Part 5: Compare All Models Across Datasets

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split

# 📁 Kaggle dataset: https://www.kaggle.com/datasets/walimuhammadahmad/fakeaudio
# Set the dataset root folder via env var, e.g.
#   export FAKEAUDIO_DIR="/path/to/fakeaudio"
# If not set, we try a few common local/Kaggle locations.

def resolve_fakeaudio_dir() -> Path:
    candidates = []

    env = os.environ.get("FAKEAUDIO_DIR")
    if env:
        candidates.append(Path(env))

    candidates += [
        Path("data/fakeaudio"),
        Path("fakeaudio"),
        Path("/kaggle/input/fakeaudio"),
        Path("/kaggle/input/walimuhammadahmad-fakeaudio"),
    ]

    for p in candidates:
        if p.exists() and p.is_dir():
            return p

    raise FileNotFoundError(
        "FakeAudio dataset folder not found. "
        "Download/unzip the dataset and set FAKEAUDIO_DIR to its root folder (the one containing real/fake or bonafide/spoof subfolders)."
    )


FAKEAUDIO_DIR = resolve_fakeaudio_dir()

# 📁 Dataset roots (can add more later)
dataset_dirs = {
    "fakeaudio": str(FAKEAUDIO_DIR)
}


def _pick_split_dir(root: Path, candidates: list[str]) -> Path | None:
    for c in candidates:
        p = root / c
        if p.exists() and p.is_dir():
            return p
    return None


def _get_splits(dataset_root: Path):
    """Return (train_dir, val_dir, test_dir) or (None, None, None) if not present."""
    train_dir = _pick_split_dir(dataset_root, ["train", "training"])
    val_dir = _pick_split_dir(dataset_root, ["val", "valid", "validation"])
    test_dir = _pick_split_dir(dataset_root, ["test", "testing"])

    if train_dir and (val_dir or test_dir):
        return train_dir, val_dir, test_dir

    # Some datasets are directly: root/real and root/fake (no explicit splits)
    return None, None, None


all_results = {}

for name, dataset_dir in dataset_dirs.items():
    dataset_root = Path(dataset_dir)

    train_dir, val_dir, test_dir = _get_splits(dataset_root)

    if train_dir is None:
        # No predefined splits: load everything and create a split.
        X_raw, y = preprocess_audio_dir(str(dataset_root))
        X = extract_mfcc_features(X_raw)

        X_train, X_temp, y_train, y_temp = train_test_split(
            X, y, test_size=0.30, random_state=42, stratify=y
        )
        X_val, X_test, y_val, y_test = train_test_split(
            X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
        )

        X_train, X_val, X_test, _ = normalize_features(X_train, X_val, X_test)
        models = train_models(X_train, y_train)

        for model_name, model in models.items():
            filename = f"{name}_{model_name.replace(' ', '_')}.pkl"
            joblib.dump(model, filename)
            print(f"Saved model: {filename}")

        results = evaluate_models(models, X_test, y_test)
        all_results[name] = results
        continue

    # Predefined split folders
    train_path = str(train_dir)
    val_path = str(val_dir) if val_dir else None
    test_path = str(test_dir) if test_dir else None

    X_train_raw, y_train = preprocess_audio_dir(train_path)

    if val_path:
        X_val_raw, y_val = preprocess_audio_dir(val_path)
    else:
        X_val_raw, y_val = [], []

    if test_path:
        X_test_raw, y_test = preprocess_audio_dir(test_path)
    else:
        # If there's no explicit test split, use a split from train
        X_train_raw, X_test_raw, y_train, y_test = train_test_split(
            X_train_raw, y_train, test_size=0.20, random_state=42, stratify=y_train
        )

    X_train = extract_mfcc_features(X_train_raw)
    X_val = extract_mfcc_features(X_val_raw) if len(X_val_raw) else np.empty((0, 40))
    X_test = extract_mfcc_features(X_test_raw)

    X_train, X_val, X_test, _ = normalize_features(X_train, X_val, X_test)
    models = train_models(X_train, y_train)

    for model_name, model in models.items():
        filename = f"{name}_{model_name.replace(' ', '_')}.pkl"
        joblib.dump(model, filename)
        print(f"Saved model: {filename}")

    results = evaluate_models(models, X_test, y_test)
    all_results[name] = results


results = all_results
final_df = pd.DataFrame({ds: res for ds, res in results.items()}).T
display(final_df)


In [ ]:
# Flatten nested dictionary into a long-form DataFrame
flat_results = []

for dataset, model_metrics in results.items():
    for model_name, metrics in model_metrics.items():
        result_row = {
            "Dataset": dataset,
            "Model": model_name,
            "Accuracy": metrics["accuracy"],
            "Precision": metrics["precision"],
            "Recall": metrics["recall"],
            "F1 Score": metrics["f1"]
        }
        flat_results.append(result_row)

# Save and display the results
results_df = pd.DataFrame(flat_results)
results_df.to_csv("model_evaluation_results.csv", index=False)

print("\n📄 Model Evaluation Results:")
display(results_df)


In [ ]:
import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np

def plot_audio_spectrograms(audio_path, title_prefix=''):
    # Load audio
    y, sr = librosa.load(audio_path, sr=None)

    # Compute spectrogram (linear)
    D = np.abs(librosa.stft(y))

    # Convert to dB scale
    DB = librosa.amplitude_to_db(D, ref=np.max)

    # Compute log-mel spectrogram
    mel_spect = librosa.feature.melspectrogram(y=y, sr=sr)
    mel_db = librosa.power_to_db(mel_spect, ref=np.max)

    # Plot
    plt.figure(figsize=(12, 8))

    # Linear spectrogram
    plt.subplot(2, 2, 1)
    librosa.display.specshow(DB, sr=sr, x_axis='time', y_axis='linear')
    plt.title(f'{title_prefix} Spectrogram (Linear)')
    plt.colorbar(format='%+2.0f dB')

    # Log-mel spectrogram
    plt.subplot(2, 2, 2)
    librosa.display.specshow(mel_db, sr=sr, x_axis='time', y_axis='mel')
    plt.title(f'{title_prefix} Spectrogram (Mel dB)')
    plt.colorbar(format='%+2.0f dB')

    # MFCC
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    plt.subplot(2, 2, 3)
    librosa.display.specshow(mfccs, x_axis='time', sr=sr)
    plt.title(f'{title_prefix} MFCC')
    plt.colorbar()

    # Waveform
    plt.subplot(2, 2, 4)
    librosa.display.waveshow(y, sr=sr)
    plt.title(f'{title_prefix} Waveform')

    plt.tight_layout()
    plt.show()

import os
from pathlib import Path


def resolve_fakeaudio_dir() -> Path:
    candidates = []

    env = os.environ.get("FAKEAUDIO_DIR")
    if env:
        candidates.append(Path(env))

    candidates += [
        Path("data/fakeaudio"),
        Path("fakeaudio"),
        Path("/kaggle/input/fakeaudio"),
        Path("/kaggle/input/walimuhammadahmad-fakeaudio"),
    ]

    for p in candidates:
        if p.exists() and p.is_dir():
            return p

    raise FileNotFoundError(
        "FakeAudio dataset folder not found. Set FAKEAUDIO_DIR to the dataset root folder."
    )


FAKEAUDIO_DIR = resolve_fakeaudio_dir()


def _infer_label(parts_lower):
    for p in parts_lower:
        if p in {"real", "bonafide", "bona-fide", "genuine", "original"}:
            return 0
        if p in {"fake", "spoof", "deepfake", "synth", "synthetic", "generated"}:
            return 1
    return None


def find_example_audio(dataset_root: Path, want_label: int, exts=(".wav", ".flac", ".mp3", ".ogg", ".m4a")):
    for root, _, files in os.walk(dataset_root):
        for fn in files:
            if Path(fn).suffix.lower() not in exts:
                continue
            p = Path(root) / fn
            parts = [x.lower() for x in p.parts]
            if _infer_label(parts) == want_label:
                return str(p)
    return None


real_example = find_example_audio(FAKEAUDIO_DIR, 0)
fake_example = find_example_audio(FAKEAUDIO_DIR, 1)

if real_example:
    plot_audio_spectrograms(real_example, title_prefix="Real Audio")
else:
    print("No real/bonafide example audio found under", str(FAKEAUDIO_DIR))

if fake_example:
    plot_audio_spectrograms(fake_example, title_prefix="Fake Audio")
else:
    print("No fake/spoof example audio found under", str(FAKEAUDIO_DIR))



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, log_loss
from sklearn.preprocessing import LabelBinarizer

# Initialize model with warm_start=True to continue training over epochs
model = MLPClassifier(hidden_layer_sizes=(100,), max_iter=1, warm_start=True, random_state=42)

# Prepare one-hot labels for log loss
lb = LabelBinarizer()
y_train_bin = lb.fit_transform(y_train)
y_test_bin = lb.transform(y_test)

# Lists to store accuracy and loss values
train_acc, val_acc = [], []
train_loss, val_loss = [], []

n_epochs = 50

for epoch in range(n_epochs):
    model.fit(X_train, y_train)  # Continues training from previous weights

    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    y_train_proba = model.predict_proba(X_train)
    y_test_proba = model.predict_proba(X_test)

    # Accuracy
    train_acc.append(accuracy_score(y_train, y_train_pred))
    val_acc.append(accuracy_score(y_test, y_test_pred))

    # Loss
    train_loss.append(log_loss(y_train_bin, y_train_proba))
    val_loss.append(log_loss(y_test_bin, y_test_proba))

# Plotting
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(train_acc, label="Train Accuracy")
plt.plot(val_acc, label="Validation Accuracy")
plt.title("(a) Model Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy Value")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(train_loss, label="Train Loss")
plt.plot(val_loss, label="Validation Loss")
plt.title("(b) Model Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss Value")
plt.legend()

plt.tight_layout()
plt.show()


# Hyper Paramterization

In [ ]:
import os
from pathlib import Path
import numpy as np
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from scipy.stats import uniform


# 📁 Kaggle dataset: https://www.kaggle.com/datasets/walimuhammadahmad/fakeaudio

def resolve_fakeaudio_dir() -> Path:
    candidates = []

    env = os.environ.get("FAKEAUDIO_DIR")
    if env:
        candidates.append(Path(env))

    candidates += [
        Path("data/fakeaudio"),
        Path("fakeaudio"),
        Path("/kaggle/input/fakeaudio"),
        Path("/kaggle/input/walimuhammadahmad-fakeaudio"),
    ]

    for p in candidates:
        if p.exists() and p.is_dir():
            return p

    raise FileNotFoundError(
        "FakeAudio dataset folder not found. "
        "Download/unzip the dataset and set FAKEAUDIO_DIR to its root folder."
    )


FAKEAUDIO_DIR = resolve_fakeaudio_dir()

dataset_dirs = {
    "fakeaudio": str(FAKEAUDIO_DIR)
}


def _pick_split_dir(root: Path, candidates: list[str]) -> Path | None:
    for c in candidates:
        p = root / c
        if p.exists() and p.is_dir():
            return p
    return None


def _get_train_test(dataset_root: Path):
    train_dir = _pick_split_dir(dataset_root, ["train", "training"])
    test_dir = _pick_split_dir(dataset_root, ["test", "testing"])
    if train_dir and test_dir:
        return train_dir, test_dir
    return None, None


all_results = []

for name, dataset_dir in dataset_dirs.items():
    dataset_root = Path(dataset_dir)
    train_dir, test_dir = _get_train_test(dataset_root)

    if train_dir is None:
        X_raw, y = preprocess_audio_dir(str(dataset_root))
        X = extract_mfcc_features(X_raw)
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.20, random_state=42, stratify=y
        )
        X_train, _, X_test, _ = normalize_features(X_train, np.empty((0, X_train.shape[1])), X_test)
    else:
        X_train_raw, y_train = preprocess_audio_dir(str(train_dir))
        X_test_raw, y_test = preprocess_audio_dir(str(test_dir))

        X_train = extract_mfcc_features(X_train_raw)
        X_test = extract_mfcc_features(X_test_raw)
        X_train, _, X_test, _ = normalize_features(X_train, np.empty((0, X_train.shape[1])), X_test)

    param_dist = {
        "hidden_layer_sizes": [(128,), (64, 32), (128, 64, 32)],
        "activation": ["relu", "tanh"],
        "solver": ["adam"],
        "alpha": uniform(1e-5, 1e-2),
        "learning_rate_init": uniform(1e-4, 1e-2),
        "batch_size": [32, 64, 128],
        "learning_rate": ["constant", "adaptive"],
        "max_iter": [500],
    }

    mlp = MLPClassifier(random_state=42)

    random_search = RandomizedSearchCV(
        mlp,
        param_distributions=param_dist,
        n_iter=30,
        cv=3,
        scoring="accuracy",
        n_jobs=-1,
        verbose=2,
    )

    random_search.fit(X_train, y_train)
    print("Best params:", random_search.best_params_)
    y_pred = random_search.best_estimator_.predict(X_test)

    from sklearn.metrics import accuracy_score

    test_accuracy = accuracy_score(y_test, y_pred)
    print("Test accuracy:", test_accuracy)
    all_results.append([name, test_accuracy])


In [ ]:
print("Best params:", random_search.best_params_)

In [ ]:
# Predict on test data
y_pred = random_search.best_estimator_.predict(X_test)

# Calculate accuracy
from sklearn.metrics import accuracy_score
test_accuracy = accuracy_score(y_test, y_pred)

print("Test accuracy:", test_accuracy)

# Deep Learning

In [ ]:
import os
from pathlib import Path
import librosa
import numpy as np
import hashlib
import matplotlib.pyplot as plt
from tqdm import tqdm
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Dense, Dropout, Flatten, Conv2D, MaxPooling2D, LSTM, Input
from tensorflow.keras.applications import VGG16
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import pandas as pd
import joblib

# Preprocessing functions (reusing your existing code)
def compute_md5(file_path):
    with open(file_path, 'rb') as f:
        return hashlib.md5(f.read()).hexdigest()

def load_and_pad(file_path, sr=16000):
    try:
        y, _ = librosa.load(file_path, sr=sr)
        if len(y) < sr:
            y = np.pad(y, (0, sr - len(y)))
        return y
    except:
        return None

def _infer_label_from_path(path_parts_lower):
    for p in path_parts_lower:
        if p in {"real", "bonafide", "bona-fide", "genuine", "original"}:
            return 0
        if p in {"fake", "spoof", "deepfake", "synth", "synthetic", "generated"}:
            return 1
    return None


def preprocess_audio_dir(base_path, sr=16000, audio_exts=(".wav", ".flac", ".mp3", ".ogg", ".m4a")):
    all_audio = []
    all_labels = []
    seen_hashes = set()

    audio_files = []
    for root, _, files in os.walk(base_path):
        for fn in files:
            if os.path.splitext(fn)[1].lower() in audio_exts:
                audio_files.append(os.path.join(root, fn))

    if not audio_files:
        raise FileNotFoundError(
            f"No audio files found under: {base_path}. "
            "Check FAKEAUDIO_DIR (it should point to the dataset root folder)."
        )

    for path in tqdm(audio_files, desc=f"Scanning {os.path.basename(str(base_path).rstrip('/')) or base_path}"):
        try:
            if os.path.getsize(path) == 0:
                continue
        except OSError:
            continue

        parts = [p.lower() for p in os.path.normpath(path).split(os.sep)]
        label = _infer_label_from_path(parts)
        if label is None:
            continue

        file_hash = compute_md5(path)
        if file_hash in seen_hashes:
            continue
        seen_hashes.add(file_hash)

        y = load_and_pad(path, sr=sr)
        if y is None:
            continue

        all_audio.append(y)
        all_labels.append(label)

    return all_audio, all_labels

# Enhanced feature extraction for deep learning (generating spectrograms)
def extract_mfcc_spectrograms(audio_list, sr=16000, n_mfcc=40):
    """Extract MFCC spectrograms for deep learning models"""
    spectrograms = []
    for y in tqdm(audio_list, desc="Extracting MFCC Spectrograms"):
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
        # Normalize mfcc features per audio file
        mfcc = (mfcc - np.mean(mfcc)) / (np.std(mfcc) + 1e-8)
        # Pad or trim to fixed size (e.g., 40 x 64)
        if mfcc.shape[1] < 64:
            pad_width = 64 - mfcc.shape[1]
            mfcc = np.pad(mfcc, ((0, 0), (0, pad_width)))
        else:
            mfcc = mfcc[:, :64]
        spectrograms.append(mfcc)
    
    # Convert to numpy array and reshape for CNN (add channel dimension)
    spectrograms = np.array(spectrograms)
    spectrograms = np.expand_dims(spectrograms, axis=-1)  # Shape: (n_samples, n_mfcc, time_steps, 1)
    return spectrograms

# Create VGG16-based model for deepfake detection
def create_vgg16_model(input_shape):
    """Create a VGG16-based transfer learning model"""
    # Create a base model from VGG16 without the top layer
    base_model = VGG16(weights='imagenet', include_top=False, input_shape=(64, 64, 3))
    
    # Freeze the base model layers
    for layer in base_model.layers:
        layer.trainable = False
    
    # Create new model
    inputs = Input(shape=input_shape)
    
    # Convert grayscale to 3-channel (if input is grayscale)
    if input_shape[-1] == 1:
        x = tf.keras.layers.Conv2D(3, (1, 1), padding='same')(inputs)
    else:
        x = inputs
    
    # Resize to 64x64 if needed
    x = tf.keras.layers.Resizing(64, 64)(x)
    
    # Pass through the VGG16 model
    x = base_model(x)
    
    # Add classification layers
    x = Flatten()(x)
    x = Dense(256, activation='relu')(x)
    x = Dropout(0.5)(x)
    x = Dense(128, activation='relu')(x)
    x = Dropout(0.3)(x)
    outputs = Dense(2, activation='softmax')(x)
    
    # Create the model
    model = Model(inputs=inputs, outputs=outputs)
    
    # Compile the model
    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    return model

# Create LSTM model for deepfake detection
def create_lstm_model(input_shape):
    """Create an LSTM model for MFCC sequence analysis"""
    # Reshape input to be suitable for LSTM (samples, time steps, features)
    lstm_input_shape = (input_shape[1], input_shape[0])  # (time_steps, features)
    
    model = Sequential([
        Input(shape=input_shape),
        tf.keras.layers.Reshape(lstm_input_shape),
        LSTM(128, return_sequences=True),
        Dropout(0.2),
        LSTM(64),
        Dropout(0.2),
        Dense(32, activation='relu'),
        Dense(2, activation='softmax')
    ])
    
    # Compile the model
    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    return model

# Data preparation for VGG16
def prepare_data_for_dl(X_spectrograms, y_labels):
    """Prepare data for deep learning models"""
    # Convert labels to categorical
    y_categorical = to_categorical(y_labels, num_classes=2)
    
    return X_spectrograms, y_categorical

# Main execution
if __name__ == "__main__":
    # 📁 Kaggle dataset: https://www.kaggle.com/datasets/walimuhammadahmad/fakeaudio
    def resolve_fakeaudio_dir() -> Path:
        candidates = []

        env = os.environ.get("FAKEAUDIO_DIR")
        if env:
            candidates.append(Path(env))

        candidates += [
            Path("data/fakeaudio"),
            Path("fakeaudio"),
            Path("/kaggle/input/fakeaudio"),
            Path("/kaggle/input/walimuhammadahmad-fakeaudio"),
        ]

        for p in candidates:
            if p.exists() and p.is_dir():
                return p

        raise FileNotFoundError(
            "FakeAudio dataset folder not found. Set FAKEAUDIO_DIR to the dataset root folder."
        )

    fakeaudio_dir = resolve_fakeaudio_dir()

    dataset_dirs = {
        "fakeaudio": str(fakeaudio_dir)
    }

    def _pick_split_dir(root: Path, candidates: list[str]) -> Path | None:
        for c in candidates:
            p = root / c
            if p.exists() and p.is_dir():
                return p
        return None

    all_results = {}

    for name, dataset_dir in dataset_dirs.items():
        print(f"Processing dataset: {name}")

        dataset_root = Path(dataset_dir)
        train_dir = _pick_split_dir(dataset_root, ["train", "training"])
        val_dir = _pick_split_dir(dataset_root, ["val", "valid", "validation"])
        test_dir = _pick_split_dir(dataset_root, ["test", "testing"])

        if train_dir is None:
            # No predefined splits: load everything and create splits
            X_all_raw, y_all = preprocess_audio_dir(str(dataset_root))
            X_train_raw, X_temp_raw, y_train, y_temp = train_test_split(
                X_all_raw, y_all, test_size=0.30, random_state=42, stratify=y_all
            )
            X_val_raw, X_test_raw, y_val, y_test = train_test_split(
                X_temp_raw, y_temp, test_size=0.50, random_state=42, stratify=y_temp
            )
        else:
            X_train_raw, y_train = preprocess_audio_dir(str(train_dir))

            if val_dir:
                X_val_raw, y_val = preprocess_audio_dir(str(val_dir))
            else:
                X_val_raw, y_val = [], []

            if test_dir:
                X_test_raw, y_test = preprocess_audio_dir(str(test_dir))
            else:
                X_train_raw, X_test_raw, y_train, y_test = train_test_split(
                    X_train_raw, y_train, test_size=0.20, random_state=42, stratify=y_train
                )
                X_val_raw, y_val = [], []
        
        # Extract MFCC spectrograms for deep learning
        X_train_spectrograms = extract_mfcc_spectrograms(X_train_raw)
        X_val_spectrograms = extract_mfcc_spectrograms(X_val_raw)
        X_test_spectrograms = extract_mfcc_spectrograms(X_test_raw)
        
        # Prepare data for deep learning
        X_train_dl, y_train_dl = prepare_data_for_dl(X_train_spectrograms, y_train)
        X_val_dl, y_val_dl = prepare_data_for_dl(X_val_spectrograms, y_val)
        X_test_dl, y_test_dl = prepare_data_for_dl(X_test_spectrograms, y_test)

In [ ]:
        # Set up callbacks
        checkpoint_vgg = ModelCheckpoint(
            f"{name}_vgg16_model.keras",  # Using the .keras extension
            monitor='val_accuracy',
            save_best_only=True,
            mode='max',
            verbose=1
        )
        
        checkpoint_lstm = ModelCheckpoint(
            f"{name}_lstm_model.keras",  # Change this extension too
            monitor='val_accuracy',
            save_best_only=True,
            mode='max',
            verbose=1
        )
        
        early_stopping = EarlyStopping(
            monitor='val_loss',
            patience=5,
            restore_best_weights=True
        )
        
        # Create and train VGG16 model
        print("Training VGG16 model...")
        input_shape = X_train_spectrograms.shape[1:]
        vgg16_model = create_vgg16_model(input_shape)
        
        vgg16_history = vgg16_model.fit(
            X_train_dl,
            y_train_dl,
            validation_data=(X_val_dl, y_val_dl),
            epochs=20,
            batch_size=32,
            callbacks=[checkpoint_vgg, early_stopping]
        )
        
        # Create and train LSTM model
        print("Training LSTM model...")
        lstm_model = create_lstm_model(input_shape)
        
        lstm_history = lstm_model.fit(
            X_train_dl,
            y_train_dl,
            validation_data=(X_val_dl, y_val_dl),
            epochs=20,
            batch_size=32,
            callbacks=[checkpoint_lstm, early_stopping]
        )
        
        # Evaluate models
        vgg16_eval = vgg16_model.evaluate(X_test_dl, y_test_dl)
        lstm_eval = lstm_model.evaluate(X_test_dl, y_test_dl)
        
        # Get predictions
        vgg16_pred = np.argmax(vgg16_model.predict(X_test_dl), axis=1)
        lstm_pred = np.argmax(lstm_model.predict(X_test_dl), axis=1)
        y_test_classes = np.argmax(y_test_dl, axis=1)
        
        # Calculate metrics
        vgg16_accuracy = accuracy_score(y_test_classes, vgg16_pred)
        lstm_accuracy = accuracy_score(y_test_classes, lstm_pred)
        
        vgg16_report = classification_report(y_test_classes, vgg16_pred, output_dict=True)
        lstm_report = classification_report(y_test_classes, lstm_pred, output_dict=True)
        
        # Store results
        results = {
            "VGG16": {
                "accuracy": vgg16_accuracy,
                "precision": vgg16_report['weighted avg']['precision'],
                "recall": vgg16_report['weighted avg']['recall'],
                "f1": vgg16_report['weighted avg']['f1-score']
            },
            "LSTM": {
                "accuracy": lstm_accuracy,
                "precision": lstm_report['weighted avg']['precision'],
                "recall": lstm_report['weighted avg']['recall'],
                "f1": lstm_report['weighted avg']['f1-score']
            }
        }
        
        all_results[name] = results
        
        # Plot training history
        plt.figure(figsize=(12, 4))
        
        plt.subplot(1, 2, 1)
        plt.plot(vgg16_history.history['accuracy'], label='Train Accuracy')
        plt.plot(vgg16_history.history['val_accuracy'], label='Validation Accuracy')
        plt.title(f'VGG16 Model Accuracy - {name}')
        plt.xlabel('Epoch')
        plt.ylabel('Accuracy')
        plt.legend()
        
        plt.subplot(1, 2, 2)
        plt.plot(lstm_history.history['accuracy'], label='Train Accuracy')
        plt.plot(lstm_history.history['val_accuracy'], label='Validation Accuracy')
        plt.title(f'LSTM Model Accuracy - {name}')
        plt.xlabel('Epoch')
        plt.ylabel('Accuracy')
        plt.legend()
        
        plt.tight_layout()
        plt.savefig(f"{name}_training_history.png")
        plt.close()
        
        # Plot confusion matrices
        plt.figure(figsize=(12, 5))
        
        plt.subplot(1, 2, 1)
        vgg16_cm = confusion_matrix(y_test_classes, vgg16_pred)
        plt.imshow(vgg16_cm, cmap='Blues')
        plt.title(f'VGG16 Confusion Matrix - {name}')
        plt.colorbar()
        plt.xlabel('Predicted Label')
        plt.ylabel('True Label')
        plt.xticks([0, 1], ['Real', 'Fake'])
        plt.yticks([0, 1], ['Real', 'Fake'])
        
        for i in range(2):
            for j in range(2):
                plt.text(j, i, f"{vgg16_cm[i, j]}", ha='center', va='center')
        
        plt.subplot(1, 2, 2)
        lstm_cm = confusion_matrix(y_test_classes, lstm_pred)
        plt.imshow(lstm_cm, cmap='Blues')
        plt.title(f'LSTM Confusion Matrix - {name}')
        plt.colorbar()
        plt.xlabel('Predicted Label')
        plt.ylabel('True Label')
        plt.xticks([0, 1], ['Real', 'Fake'])
        plt.yticks([0, 1], ['Real', 'Fake'])
        
        for i in range(2):
            for j in range(2):
                plt.text(j, i, f"{lstm_cm[i, j]}", ha='center', va='center')
        
        plt.tight_layout()
        plt.savefig(f"{name}_confusion_matrices.png")
        plt.close()
    
    # Print final results as DataFrame
print("\nFinal results:")
final_df = pd.DataFrame({ds: res for ds, res in all_results.items()}).T
print(final_df)

In [ ]:
# Print final results in a more readable format
print("\nFinal results:")
for dataset_name, models in all_results.items():
    print(f"\nDataset: {dataset_name}")
    for model_name, metrics in models.items():
        print(f"  {model_name} model:")
        for metric_name, value in metrics.items():
            print(f"    {metric_name}: {value:.4f}")